# AG-HYPOPT · experiment_1 · Trial 03

**Hypothesis:** <one line: what this trial tests>

> ▶ = run the cell · ✍️ = write here before continuing · ⛔ never "Run All" | full rules: `instructions.md`


## 1. ✍️ Read and summarize

Read `context.md` (physics/model knowledge) and `trials.json` (the registry). Read **all
previous executed trial notebooks** in this folder, i.e. the `trial_XX` notebooks numbered
lower than this trial.

At `trial_01` there are no previous trials yet: the campaign's starting history is the
`baseline_17g` entry in `trials.json`.

Then write below your current understanding of the situation: where the campaign stands, what
the previous trials showed, and what this trial should test.

<agent writes here>


In [ ]:
# 2. ▶ Propose candidate trials (AGHyperopt)
# API contract: AGHyperopt is implemented in ag_hypopt.py. fit() reads the space and the trial
# history; propose_trials() prints a candidate table (ID | EI | explore | params) and returns
# the candidates for the cells below.
import os

EXPERIMENT_DIR = os.getcwd()                     # notebooks run in place from the experiment folder
SPACE_PATH = os.path.join(EXPERIMENT_DIR, 'space.json')
TRIALS_PATH = os.path.join(EXPERIMENT_DIR, 'trials.json')

MAX_TRIALS = 10    # campaign cap: stop generating new trials after this many (None = unlimited)

from ag_hypopt import AGHyperopt

opt = AGHyperopt()
opt.fit(SPACE_PATH, TRIALS_PATH)
proposed_trials = opt.propose_trials(10)


## 3. ✍️ Analyze and choose

Analyze the proposed trials using **physics reasoning** (`context.md` failure modes) and the
history you summarized in cell 1. Note any **disagreement with the EI ranking**. **Choose ONE**
and justify in detail: which failure mode it attacks, what you expect to happen, and what
would confirm or refute it.

<agent writes here>


In [ ]:
# 4. ▶ Run the chosen trial   ⛔ ~3.5 h: do not interrupt unless obviously broken
INDEX = 1                # <-- your chosen candidate (1-based, from the table in cell 2)
TRIAL_ID = 'trial_03'   # auto-stamped at generation; do not edit

assert 1 <= INDEX <= len(proposed_trials), 'bad INDEX'
CHOSEN = proposed_trials[INDEX - 1]['params']
print('chosen:', CHOSEN)

# API contract: run_trial / compute_objective / format_report come from ag_hypopt (next session).
from ag_hypopt import run_trial, compute_objective, format_report

import traceback, time
t0 = time.time()
try:
    results = run_trial(CHOSEN)
    print(f'trial finished in {(time.time()-t0)/60:.1f} min')
    objective, uncertainty, breakdown = compute_objective(results)
    print(f'objective (MSE vs true) = {objective:.4f} ± {uncertainty:.4f}')
    print(format_report(breakdown))
except Exception:
    traceback.print_exc()
    results, objective, uncertainty = None, None, None


## 5. ✍️ Analyze the results, write analysis and summary

Write a comprehensive analysis: what happened vs expectations, **hypothesis confirmed or
refuted** (evidence, not vibes), what was learned about the physics and the hyperparameters.

End the analysis with the two lines that go into `trials.json`:

**Summary:** <one-line summary of this trial>
**Key insight:** <one sentence>

Structural-change proposals (score, likelihood, model, protocol) belong here as notes for
Anuar: describe them, never act on them, and never stop the campaign because of them.

<agent writes here>


In [ ]:
# 6. ▶ Record the trial in trials.json  (fill the two strings below first; copy them from cell 5)
import json

SUMMARY = '<one-line summary of this trial, from your analysis above>'
KEY_INSIGHT = '<one sentence>'

entry = {
    'trial_id': TRIAL_ID,
    'config': CHOSEN,
    'objective': objective,
    'uncertainty': uncertainty,
    'summary': SUMMARY,
    'key_insight': KEY_INSIGHT,
    'notebook': f'{TRIAL_ID}.ipynb',
}
data = json.load(open(TRIALS_PATH))
assert not any(t.get('trial_id') == TRIAL_ID for t in data['trials']),     f'{TRIAL_ID} is already registered in trials.json'
data['trials'].append(entry)
json.dump(data, open(TRIALS_PATH, 'w'), indent=2)
best = min((t for t in data['trials'] if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print('saved', TRIAL_ID, '| current best:', best['trial_id'] if best else None)


In [ ]:
# 7. ▶ Generate the next trial (or end the campaign)
import os, re, json as _json

num = int(re.search(r'(\d+)$', TRIAL_ID).group(1))
nxt = num + 1
if MAX_TRIALS is not None and nxt > MAX_TRIALS:
    print(f'Campaign complete: cap MAX_TRIALS={MAX_TRIALS} reached after {TRIAL_ID}. Stop here.')
else:
    template_path = os.path.join(EXPERIMENT_DIR, 'template.ipynb')
    target = os.path.join(EXPERIMENT_DIR, f'trial_{nxt:02d}.ipynb')
    assert not os.path.exists(target), f'{target} already exists: refusing to overwrite'

    nb = _json.load(open(template_path))

    def _stamp(old, new):
        for c in nb['cells']:
            if old in ''.join(c.get('source', [])):
                c['source'] = [s.replace(old, new) for s in c['source']]
                return True
        raise RuntimeError(f'stamp target {old!r} not found in the template')

    _stamp('{N}', f'{nxt:02d}')
    _stamp('trial_XXX', f'trial_{nxt:02d}')
    _json.dump(nb, open(target, 'w'), indent=1)
    print(f'Created {os.path.basename(target)}. Open it and follow its cells from the top.')
